# Week 3, day 3 (afternoon) — Worksheet 02 SOLUTIONS: stages and COPY INTO

The six lab steps, with the reasoning behind each choice the lab makes.

**Not executed.** The SQL comes from the lab scripts in `snowflake-scripts/`; the
row counts quoted are the ones Worksheets 03 to 05 measured on the same two CSVs
locally, so those specific figures — 1,215 and 100,000 — are observed. Everything
about Snowflake's behaviour is from the documentation and the lab, and is yours
to confirm.

PART A — environment and stages

### Question 1

Run `1_environment_setup.sql`. Then run `SHOW SCHEMAS IN DATABASE LABDB;` and record every schema it lists — including the ones you did not create.
> **NOTE:** the script creates two schemas. `SHOW SCHEMAS` will return more than two.

```sql
CREATE DATABASE IF NOT EXISTS LABDB;
USE DATABASE LABDB;
CREATE SCHEMA IF NOT EXISTS LABDB.RAW;
CREATE SCHEMA IF NOT EXISTS LABDB.CORE;

SHOW SCHEMAS IN DATABASE LABDB;
```

Four schemas, not two: `RAW` and `CORE` as created, plus `PUBLIC` and
`INFORMATION_SCHEMA`, which every Snowflake database gets automatically.
`INFORMATION_SCHEMA` is the metadata catalogue — queryable, and the answer to
"what tables exist and what is in them" when you need it programmatically rather
than by clicking.

The split between `RAW` and `CORE` is the part worth taking seriously. `RAW` holds
data as it arrived, unmodified and append-only. `CORE` holds the modelled version
— conformed, deduplicated, joined. Nothing writes to `CORE` except transformations
that read from `RAW`, so the raw layer stays a faithful record of what the source
sent.

That is what makes a pipeline re-runnable. When a transformation turns out to be
wrong — and Worksheets 03 to 05 find several reasons it might be — you fix the
transformation and rebuild `CORE` from `RAW`. If the cleaning happened on the way
in, the only way back is to ask the source system for the files again, and it may
not have them.

`IF NOT EXISTS` throughout, so the script can be re-run without destroying
anything. That theme continues in Q5.

### Question 2

Run `2_create_internal_stage.sql` to create `PRODUCTS_STAGE` and `SALES_STAGE`. Then run `SHOW STAGES IN SCHEMA LABDB.RAW;` and record the `type` column. Explain what a *named internal* stage is, and why the lab uses one rather than a table stage (`@%STG_Sales`) or a user stage (`@~`).

```sql
USE LABDB;
USE SCHEMA RAW;

CREATE STAGE products_stage;
CREATE STAGE sales_stage;

SHOW STAGES IN SCHEMA LABDB.RAW;
```

Two stages, `type` = `INTERNAL`.

A **stage** is a staging area between your files and your tables. You put a file
in the stage, then `COPY INTO` reads it from there. The indirection is the point:
once a file is staged, you can inspect it (Q4), load it into more than one table,
load a subset of it, or reload it after fixing a `FILE_FORMAT` — without moving
the bytes again.

**Internal** stages live in Snowflake's own storage; **external** stages point at
your S3, GCS, or Azure bucket. Internal has three flavours:

| kind | reference | scope |
|---|---|---|
| table stage | `@%STG_Sales` | one table, auto-created |
| user stage | `@~` | one user, auto-created |
| named internal | `@PRODUCTS_STAGE` | explicit, shareable |

The lab uses named internal for a practical reason the lab states outright: table
and user stages **cannot be uploaded to through the Snowsight UI**. Loading to
them needs `PUT` from SnowSQL or a driver — a command-line tool you would have to
install. Named stages support UI upload.

There is a better reason too. A table stage belongs to one table, so it is a dead
end the moment two tables need the same file, or you want to keep the file after
dropping the table. A named stage is an object in the schema with its own grants,
its own lifecycle, and its own `FILE_FORMAT` if you want one. That is why real
pipelines use them.

### Question 3

Upload the files through **Ingestion → Add data → Load files into a Stage**: `data/products_2013_01_01.csv` into `PRODUCTS_STAGE`, `data/sales_2013_01_01.csv` into `SALES_STAGE`, both with path `/data_loading_lab/csv_files`. Then run `LIST @LABDB.RAW.PRODUCTS_STAGE;` and `LIST @LABDB.RAW.SALES_STAGE;` and record the full `name` and `size` of each file.
> **NOTE:** compare the size Snowflake reports with the size on disk. They will not match.

```sql
LIST @LABDB.RAW.PRODUCTS_STAGE;
LIST @LABDB.RAW.SALES_STAGE;
```

Two rows, one per stage, with `name` giving the full path inside the stage:

```
products_stage/data_loading_lab/csv_files/products_2013_01_01.csv
sales_stage/data_loading_lab/csv_files/sales_2013_01_01.csv
```

That path is what `METADATA$FILENAME` returns during the COPY, and it is why the
lab's `SPLIT_PART(METADATA$FILENAME, '/', -1)` takes the **last** segment to get
just the filename. Worksheet 04 Q9 rebuilds that expression.

The path is also not decoration. `COPY INTO ... FROM @SALES_STAGE/2024/01/` loads
every file under that prefix — so the folder structure you choose when uploading
becomes the unit you can reload later. `/data_loading_lab/csv_files` is a
reasonable habit even for one file.

**On the sizes:** the numbers Snowflake reports are smaller than the files on
disk, because internal stages compress on upload (gzip by default) and `LIST`
reports the stored size. `COPY INTO` decompresses transparently, so nothing about
your load changes.

Which is worth knowing for a specific reason: a stage size that looks
suspiciously small is normal, and a stage size that matches the disk size exactly
usually means the file was already compressed. Do not use `LIST` sizes to check
that a file uploaded completely — use the row count after loading.

### Question 4

Before loading anything, query the staged file directly: `SELECT $1, $2, $6, $13 FROM @LABDB.RAW.SALES_STAGE/data_loading_lab/csv_files/sales_2013_01_01.csv LIMIT 5;`. Record what comes back, paying attention to `$6` and `$13`.
> **NOTE:** no `FILE_FORMAT` clause here, so no `SKIP_HEADER` and no quote handling. That is the point.

```sql
SELECT $1, $2, $6, $13
FROM @LABDB.RAW.SALES_STAGE/data_loading_lab/csv_files/sales_2013_01_01.csv
LIMIT 5;
```

Five rows, and the first one is the **header**: `$1` is the literal text
`TRANS_ID`, `$6` is `PRIORITY`, `$13` is `SHIPMODE`. With no `FILE_FORMAT` clause
there is no `SKIP_HEADER`, so the header is just the first line of the file.

Then rows 2 to 5, where `$6` and `$13` come back **with quotes attached** —
`"Medium"`, `"Regular Air"` — because there is no `FIELD_OPTIONALLY_ENCLOSED_BY`
either. Worksheet 03 Q1 shows why: the file has three quote characters a side, so
even with the setting one pair survives.

Note `$13`. The header calls that column `SHIPMODE`; `STG_Sales` declares
`SHIP_MODE`. Querying by position sidesteps the question entirely — `$13` is the
thirteenth field whatever anyone calls it — which is exactly how the `COPY INTO`
works and exactly why the mismatch never surfaces. Worksheet 03 Q10 shows what
happens when something matches by name instead.

**Querying a stage directly is the habit to take from this question.** It costs
nothing, needs no table, and answers "what is actually in this file" before you
have committed to a schema. Given an unfamiliar CSV, this is the first thing to
run — not the `CREATE TABLE`.

PART B — tables and the load

### Question 5

Run the `CREATE TABLE` half of `3_Stage_tables.sql`. Then run `DESC TABLE LABDB.RAW.STG_Sales;` and record the 16 columns. Explain why the lab uses `CREATE TABLE IF NOT EXISTS` rather than `CREATE OR REPLACE`, and why `INSERTED_AT` is not in the COPY's column list.

```sql
DESC TABLE LABDB.RAW.STG_Sales;
```

Sixteen columns: the fourteen from the file, plus `BATCH_ID VARCHAR(50)` and
`INSERTED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP()`.

**`CREATE TABLE IF NOT EXISTS` rather than `CREATE OR REPLACE`.** `CREATE OR
REPLACE` drops the existing table and builds a new one — silently, with no
confirmation, taking every row with it. In `RAW`, which is append-only and holds
the historical load record, that is the one thing you must not do. `IF NOT
EXISTS` makes the script safe to re-run: it creates the table the first time and
does nothing every time after.

The trade-off is real and worth naming. `IF NOT EXISTS` will also silently do
nothing when the table exists with the *wrong* schema — so a column you added to
the script never appears, and the next COPY fails with a confusing error. The
answer is not to switch to `CREATE OR REPLACE`; it is `ALTER TABLE`, deliberately.

**`INSERTED_AT` is omitted from the COPY's column list** because it has a
`DEFAULT`. Snowflake populates it automatically for every row the COPY inserts.
Listing it would mean supplying a value, and the file has none to supply.

The two metadata columns answer different questions and you need both.
`INSERTED_AT` is when the row **arrived**; `TRANS_DT` is when the event
**happened**. In this data they are four years apart. Confusing the two is how
"sales in the last 7 days" quietly becomes "rows loaded in the last 7 days" —
which returns everything on the day of a backfill.

### Question 6

Run both `COPY INTO` statements from `3_Stage_tables.sql`. Record the full result grid for each — `file`, `status`, `rows_parsed`, `rows_loaded`, `errors_seen`. Then compare `rows_parsed` with `rows_loaded` and say what a difference between them would mean.

Each `COPY INTO` returns one row per file loaded:

```
file                                                          status  rows_parsed  rows_loaded  errors_seen
.../products_2013_01_01.csv                                   LOADED         1215         1215            0
.../sales_2013_01_01.csv                                      LOADED       100000       100000            0
```

**1,215 and 100,000** — and those two figures are observed, not claimed:
Worksheet 05 Q1 counts exactly those rows in the same two files locally. If your
COPY reports different numbers, something is wrong with the upload, not with the
expectation.

`rows_parsed` is what Snowflake read from the file; `rows_loaded` is what reached
the table. When they differ, rows were rejected — a type conversion that failed,
a row with the wrong number of fields — and `errors_seen` counts them. Here they
are equal, so nothing was dropped.

Note that `rows_parsed` is 100,000 and not 100,001: `SKIP_HEADER = 1` removes the
header before parsing begins.

The default `ON_ERROR` for `COPY INTO` is `ABORT_STATEMENT` — one bad row and the
whole load rolls back, which is usually what you want in `RAW`. The alternatives
are `CONTINUE` (load what parses, skip the rest) and `SKIP_FILE`. `CONTINUE` is
the dangerous one: it turns a loud failure into a `rows_loaded` that is quietly
lower than `rows_parsed`, and nothing downstream will tell you rows are missing.

Which is the habit: **read `rows_loaded`, not `status`.** `LOADED` with
`ON_ERROR = CONTINUE` and 400 errors is still `LOADED`.

### Question 7

Run the two `COPY INTO` statements **a second time**, unchanged. Record what comes back, then run `SELECT COUNT(*) FROM LABDB.RAW.STG_Sales;` and confirm it did not change. Explain the mechanism, and name two things that would defeat it.
> **NOTE:** this is the most important question on the sheet. Re-running a load is not a hypothetical; it is what happens when a pipeline retries.

The second run returns:

```
Copy executed with 0 files processed.
```

and `SELECT COUNT(*)` still reports 100,000. Nothing was loaded, nothing was
duplicated.

**The mechanism is load metadata.** Snowflake records, per target table, which
files it has already loaded — by path, size, and ETag — and keeps that record for
**64 days**. A `COPY INTO` skips any file already in that record. This is what
makes the statement safe to retry, which matters because retrying is what
pipelines do: a task times out, an orchestrator re-runs the step, and without
this you would have doubled the table.

Worksheet 05 Q2 shows what a doubled load looks like from the other side —
200,000 rows under one `BATCH_ID`, with `COUNT(DISTINCT TRANS_ID)` unchanged at
7,916, so any report built on distinct counts shows nothing wrong while every
`SUM` has doubled.

**Things that defeat it** — any two of these are a fine answer:

- `FORCE = TRUE` in the COPY, which ignores the metadata entirely. It exists for
  deliberate reloads and it is the fastest way to duplicate a table by accident.
- **Recreating the stage.** `CREATE OR REPLACE STAGE` gives a new stage object,
  so the previously-loaded paths no longer match and every file loads again.
- **Re-uploading under a different path.** The record is per path;
  `/csv_files/sales.csv` and `/csv_files_v2/sales.csv` are two different files as
  far as the metadata is concerned, even byte-identical.
- **Waiting more than 64 days.** The record expires. A monthly re-run that loads
  from a long-lived stage will eventually re-load old files.
- **A new target table.** The metadata is per table, so `COPY INTO STG_SALES_2`
  from the same stage loads the file again — which is exactly what Q10 does.

So: protection by default, defeated by several ordinary operations. It is a good
safety net and a bad primary control. The primary control is the row count you
check afterwards.

### Question 8

Look at what actually landed. Run `SELECT DISTINCT PRIORITY FROM LABDB.RAW.STG_Sales;` and `SELECT DISTINCT SHIP_MODE FROM LABDB.RAW.STG_Sales;`, then `SELECT COUNT(*) FROM LABDB.RAW.STG_Sales WHERE PRIORITY = 'High';`. Record all three, and write the `COPY INTO` change that would fix what you find.
> **NOTE:** Worksheet 03 Q5 predicts the answer to the third query. Check it against what Snowflake gives you.

```sql
SELECT DISTINCT PRIORITY FROM LABDB.RAW.STG_Sales;
SELECT DISTINCT SHIP_MODE FROM LABDB.RAW.STG_Sales;
SELECT COUNT(*) FROM LABDB.RAW.STG_Sales WHERE PRIORITY = 'High';
```

Five priorities and three ship modes, every one of them still carrying a pair of
quote characters — `"Critical"`, `"High"`, `"Low"`, `"Medium"`, `"Not Specified"`;
`"Delivery Truck"`, `"Express Air"`, `"Regular Air"`.

And the count returns **0**.

Worksheet 03 Q5 measured the same thing locally: zero rows match `'High'`, while
**21,117** match `'"High"'`. The rows are there. The query cannot see them.

`FIELD_OPTIONALLY_ENCLOSED_BY = '"'` did its job — it stripped one pair. The file
has three quotes a side (Worksheet 03 Q1), so one pair remains as data.

**The fix goes in the transformational SELECT**, where the COPY is already
selecting positionally:

```sql
FROM (
    SELECT $1, $2, $3, $4, $5,
           TRIM($6, '"'),                    -- PRIORITY
           $7, $8, $9, $10, $11, $12,
           TRIM($13, '"'),                   -- SHIP_MODE
           $14,
           SPLIT_PART(METADATA$FILENAME, '/', -1)
    FROM @"SALES_STAGE"/data_loading_lab/csv_files/sales_2013_01_01.csv
)
```

Once, at load time. The alternative is `WHERE PRIORITY = '"High"'` in every query
anyone ever writes against this table, and eventually somebody writes the obvious
version and gets a confident, empty, wrong answer.

Note that the load metadata from Q7 will now stop you re-running this. Either
`FORCE = TRUE` or truncate the table first — and in `RAW`, truncating is a
decision, not a keystroke.

**The general point:** a `SELECT DISTINCT` on every low-cardinality text column,
immediately after loading, costs seconds and catches the entire class of problem
where the load succeeded and the values are wrong.

PART C — validating, and the alternate path

### Question 9

Run `4_Validate_stage_tables.sql`. Record the row counts. Then compare its Step 2 query with the version printed in the lab page — one groups by `BATCH_ID, INSERTED_AT`, the other by `BATCH_ID` alone with `MIN(INSERTED_AT)`. Say which you would keep and why.

```sql
SELECT BATCH_ID, INSERTED_AT, COUNT(*) AS row_count
FROM STG_Products GROUP BY BATCH_ID, INSERTED_AT;

SELECT BATCH_ID, INSERTED_AT AS loaded_at, COUNT(*) AS row_count
FROM STG_Sales GROUP BY BATCH_ID, INSERTED_AT;
```

One row each: `products_2013_01_01.csv` with 1,215, `sales_2013_01_01.csv` with
100,000. Both match what Worksheet 05 Q1 counts locally.

**On the difference between the two versions.** The lab page prints the second
query as:

```sql
SELECT BATCH_ID, MIN(INSERTED_AT) AS loaded_at, COUNT(*) AS row_count
FROM STG_Sales GROUP BY BATCH_ID;
```

`GROUP BY BATCH_ID` alone, with `INSERTED_AT` aggregated. The `.sql` file groups
by both columns.

Keep the lab page's version. The reason is what each one is grouping *by*:
`BATCH_ID` identifies the source file, which is the thing you want one row per.
`INSERTED_AT` is a timestamp, and grouping by a timestamp means any variation in
it splits your batch into several rows. Here it will not — Snowflake evaluates
`CURRENT_TIMESTAMP()` once per statement, so every row from one COPY carries the
same value — but the query stops being a per-file audit the moment that is not
true, and "one row per file" is what makes the output readable at all. Load ten
files and grouping by both gives you ten rows only by luck.

`MIN(INSERTED_AT)` is also the more honest label: it says *when this batch first
landed*, which is a fact about the batch. `GROUP BY INSERTED_AT` implies the
timestamp identifies something, and it does not.

Either way, note what neither version checks. Worksheets 03 to 05 find 49 missing
days, 50,039 negative margins, three money columns that disagree by 32 million,
and a join that invents 606,705.92 of revenue — all in a table that returns
exactly this clean result.

### Question 10

Two loose ends. First, Step 6: load `sales_2013_01_01.csv` again through **Add data → Load data into a Table** into a new table `STG_SALES_2`, with *Do not load any data* on error; record the row count it reports and what that table lacks compared with `STG_Sales`. Second, the last statement of `4_Validate_stage_tables.sql` — `DROP TABLE CORE.DIM_CALENDAR;` — record exactly what Snowflake says when you run it.
> **NOTE:** when you are finished, drop `STG_SALES_2` and suspend your warehouse.

**Step 6 — `Load data into a Table`.** The wizard reports **100,000 rows**
loaded into `STG_SALES_2`. Same file, same row count, no stage created, no SQL
written. For a one-off — a spreadsheet someone emailed you, a file you want to
look at once — it is genuinely the right tool, and it is worth knowing it exists.

What the table lacks is the whole difference:

- **No `BATCH_ID`.** Nothing in `STG_SALES_2` records which file its rows came
  from, so a second file loaded into it is indistinguishable from the first and
  the rows cannot be selectively removed.
- **No `INSERTED_AT`.** No record of when anything arrived.
- **No `DATE_FORMAT`.** The wizard infers types from a sample, so `TRANS_DT` is
  likely `VARCHAR` holding `3/7/2010` — and Worksheet 04 Q8 shows 38.1% of those
  strings parse to a different date under the other convention.
- **No stage.** The file is not retained, so reloading means finding it again on
  someone's laptop.
- **Nothing reproducible.** There is no artefact — no script, no diff, no review.
  The load exists as a sequence of clicks that happened once.

Note that the load metadata from Q7 did not stop this, because that record is per
*table* and `STG_SALES_2` is a new one. Same file, second copy, no warning.

**The `DROP`.** The last statement of `4_Validate_stage_tables.sql` is:

```sql
DROP TABLE CORE.DIM_CALENDAR;
```

Snowflake answers:

```
Table 'DIM_CALENDAR' does not exist or not authorized.
```

Nothing in this lab creates that table. Worksheet 05 Q10 searches all four scripts
and both lab pages: one mention, and it is this `DROP`. It is a leftover from an
environment where the table existed.

So a student who runs the four scripts in order gets a red error on the final
statement, after everything else worked — in the file named
`4_Validate_stage_tables.sql`, whose entire job is to tell them whether the load
was sound. They now cannot tell from the output whether their ingestion failed.

That is the note to end the Snowflake half on. The load worked. The validation
script is broken, in a way nobody noticed because nobody ran it end to end in a
clean account. Run yours — from an empty database, once, before you trust it.

When you are done:

```sql
DROP TABLE LABDB.RAW.STG_SALES_2;
ALTER WAREHOUSE <your_warehouse> SUSPEND;
```

Then continue to Worksheet 03, which takes these same two files apart in this
container — where every number is one you can check.